In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline



In [2]:
# --- Load dataset ---
df = pd.read_csv("D:\\Project\\Dataset EMG Fatigue\\candy_read\\data\\Athul\\features_dataset_BL.csv")
X  = df[["rms", "mav", "wl", "zcr", "mdf"]].values




In [3]:
y  = df["label"].values

In [4]:
# --- Build pipeline: scale then classify ---
# StandardScaler is critical for SVM — features are on very different scales
# (RMS in 0-500 range, MDF in 0-100 Hz range, WL in thousands)
pipeline = Pipeline([
    ("scaler",     StandardScaler()),
    ("classifier", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True))
])


In [5]:
# --- Cross validation — use StratifiedKFold to preserve class balance ---
cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores  = cross_val_score(pipeline, X, y, cv=cv, scoring="f1")
print(f"Cross-validated F1: {scores.mean():.3f} ± {scores.std():.3f}")



Cross-validated F1: 0.902 ± 0.090


In [6]:
# --- Train final model on all data ---
pipeline.fit(X, y)

# --- Save model ---
joblib.dump(pipeline, "D:\\Project\\Dataset EMG Fatigue\\candy_read\\model_train\\svm_fatigue_model.pkl")
print("Model saved to D:\\Project\\Dataset EMG Fatigue\\candy_read\\model_train\\svm_fatigue_model.pkl")



Model saved to D:\Project\Dataset EMG Fatigue\candy_read\model_train\svm_fatigue_model.pkl


In [7]:
# --- Print feature importance via permutation (optional) ---
from sklearn.inspection import permutation_importance
result = permutation_importance(pipeline, X, y, n_repeats=10, random_state=42)
feature_names = ["rms", "mav", "wl", "zcr", "mdf"]
for i in result.importances_mean.argsort()[::-1]:
    print(f"  {feature_names[i]}: {result.importances_mean[i]:.4f}")

  wl: 0.0985
  mav: 0.0687
  rms: 0.0672
  zcr: 0.0015
  mdf: -0.0060
